# Olist E-Commerce — Ingestion & Cleaning
**Day 1 Notebook** | Load all 9 CSV files, clean, merge into one master dataframe, save to CSV.

All data lives in pandas DataFrames throughout. No SQL, no Docker, no extra engines.

## 1. Imports

In [106]:
import pandas as pd
import numpy as np
from pathlib import Path

# display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('pandas version:', pd.__version__)

pandas version: 2.3.3


## 2. File Paths
Update `DATA_DIR` to point at the folder containing your 9 CSVs.

In [107]:
# --- UPDATE THIS PATH ---
DATA_DIR = Path('../rawdata')

# verify the path is correct before loading
csv_files = list(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in sorted(csv_files):
    print(' ', f.name)

Found 9 CSV files:
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_orders_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv


## 3. Load All 9 Tables
`dfs` is a **dictionary of DataFrames** — one per CSV file.
Think of it as a mini in-memory database. You reference each table as `dfs['orders']`, `dfs['customers']`, etc.
The single wide `master` DataFrame comes in Section 6 when everything is joined together.

In [108]:
FILE_MAP = {
    'orders':      'olist_orders_dataset.csv',
    'items':       'olist_order_items_dataset.csv',
    'payments':    'olist_order_payments_dataset.csv',
    'reviews':     'olist_order_reviews_dataset.csv',
    'customers':   'olist_customers_dataset.csv',
    'products':    'olist_products_dataset.csv',
    'sellers':     'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'categories':  'product_category_name_translation.csv',
}

dfs = {}
for name, fname in FILE_MAP.items():
    dfs[name] = pd.read_csv(DATA_DIR / fname)
    print(f'{name:15} {len(dfs[name]):>8,} rows   {dfs[name].shape[1]:>2} cols')

orders            99,441 rows    8 cols
items            112,650 rows    7 cols
payments         103,886 rows    5 cols
reviews           99,224 rows    7 cols
customers         99,441 rows    5 cols
products          32,951 rows    9 cols
sellers            3,095 rows    4 cols
geolocation     1,000,163 rows    5 cols
categories            71 rows    2 cols


## 4. Parse Timestamps
All date columns in `orders` come in as strings. Convert them to `datetime` now
so arithmetic (delivery delay, duration) works properly later.

In [109]:
ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# parse to datetime first
dfs['orders'][ts_cols] = dfs['orders'][ts_cols].apply(pd.to_datetime)

# then rename if you want shorter names (optional)
print(dfs['orders'].dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


## 5. Assess Missing Values
Check every table for nulls. We'll decide what to do with each one.

In [110]:
print('=== NULL COUNTS PER TABLE ===')
for name, df in dfs.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]    
    if len(nulls):
        print(f'\n{name}:')
        for col, n in nulls.items():
            pct = n / len(df) * 100
            print(f'  {col:45} {n:>6,} nulls ({pct:.1f}%)')
    else:
        print(f'\n{name}: no nulls')

=== NULL COUNTS PER TABLE ===

orders:
  order_approved_at                                160 nulls (0.2%)
  order_delivered_carrier_date                   1,783 nulls (1.8%)
  order_delivered_customer_date                  2,965 nulls (3.0%)

items: no nulls

payments: no nulls

reviews:
  review_comment_title                          87,656 nulls (88.3%)
  review_comment_message                        58,247 nulls (58.7%)

customers: no nulls

products:
  product_category_name                            610 nulls (1.9%)
  product_name_lenght                              610 nulls (1.9%)
  product_description_lenght                       610 nulls (1.9%)
  product_photos_qty                               610 nulls (1.9%)
  product_weight_g                                   2 nulls (0.0%)
  product_length_cm                                  2 nulls (0.0%)
  product_height_cm                                  2 nulls (0.0%)
  product_width_cm                                   2 nulls (0.

**Null handling:**
- `orders.order_approved_date` — ~160 nulls, cancelled orders. Leave as NaT.
- `orders.order_delivered_*` — ~2,900 nulls, not yet delivered. Filter to `delivered` status for ML.
- `reviews.review_comment_*` — ~58k nulls, optional text field. Leave, don't use for ML.
- `products.product_category_name` — ~610 nulls. Fill with `'unknown'`.

In [111]:
# only fix the one that matters for downstream joins
dfs['products']['product_category_name'] = (
    dfs['products']['product_category_name'].fillna('unknown')
)

print('product_category_name nulls remaining:',
      dfs['products']['product_category_name'].isnull().sum())

product_category_name nulls remaining: 0


## 6. Sample Geolocation
The geolocation table has 1 million rows because each zip code appears many times.
We only need one lat/lng per zip — deduplicate it down to ~70k rows.

In [112]:
print(f'Geolocation before: {len(dfs["geolocation"]):,} rows')

dfs['geolocation'] = (
    dfs['geolocation']
    .drop_duplicates(subset='geolocation_zip_code_prefix')
    .reset_index(drop=True)
)

print(f'Geolocation after:  {len(dfs["geolocation"]):,} unique zip codes')

Geolocation before: 1,000,163 rows
Geolocation after:  19,015 unique zip codes


## 7. Translate Product Categories
Category names are in Portuguese. Map them to English using the translation table.

In [113]:
cat_map = (
    dfs['categories']
    .set_index('product_category_name')['product_category_name_english']
    .to_dict()
)

dfs['products']['category_english'] = (
    dfs['products']['product_category_name']
    .map(cat_map)
    .fillna('other')
)

print('Top 10 categories:')
print(dfs['products']['category_english'].value_counts().head(10).to_string())

Top 10 categories:
category_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134


## 8. Aggregate Order Items
Each order can have multiple items. Aggregate to one row per order
before merging into the master DataFrame.

In [114]:
items_agg = (
    dfs['items']
    .groupby('order_id')
    .agg(
        order_value    = ('price',         'sum'),
        freight_value  = ('freight_value', 'sum'),
        n_items        = ('order_item_id', 'count'),
        seller_id      = ('seller_id',     'first'),
        product_id     = ('product_id',    'first'),
    )
    .reset_index()
)

print(f'items_agg shape: {items_agg.shape}')
items_agg.head(3)

items_agg shape: (98666, 6)


,order_id,order_value,freight_value,n_items,seller_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd


## 9. Aggregate Payments
Same issue — some orders have multiple payment rows (split payments).
Take the dominant payment type and sum the value.

In [115]:
payments_agg = (
    dfs['payments']
    .groupby('order_id')
    .agg(
        payment_type         = ('payment_type',         'first'),
        payment_installments = ('payment_installments', 'max'),
        payment_value        = ('payment_value',        'sum'),
    )
    .reset_index()
)

print(f'payments_agg shape: {payments_agg.shape}')
payments_agg.head(3)

payments_agg shape: (99440, 4)


,order_id,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,credit_card,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,credit_card,3,259.83
2,000229ec398224ef6ca0657da4fc703e,credit_card,5,216.87


## 10. Build the Master DataFrame
Join all tables on `order_id` and `customer_id`.
This is the single wide DataFrame that every EDA chart and ML model will read from.

In [116]:
master = (
    dfs['orders']

    # customer info (state, city)
    .merge(
        dfs['customers'][['customer_id', 'customer_state', 'customer_city']],
        on='customer_id', how='left'
    )

    # order-level revenue + item count
    .merge(items_agg, on='order_id', how='left')

    # payment info
    .merge(payments_agg, on='order_id', how='left')

    # review score
    .merge(
        dfs['reviews'][['order_id', 'review_score']],
        on='order_id', how='left'
    )

    # product category
    .merge(
        dfs['products'][['product_id', 'category_english']],
        on='product_id', how='left'
    )
)

print(f'master shape: {master.shape}')
print(f'columns: {list(master.columns)}')
master.head(3)

master shape: (99992, 20)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city', 'order_value', 'freight_value', 'n_items', 'seller_id', 'product_id', 'payment_type', 'payment_installments', 'payment_value', 'review_score', 'category_english']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,order_value,freight_value,n_items,seller_id,product_id,payment_type,payment_installments,payment_value,review_score,category_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,29.99,8.72,1.00,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,credit_card,1.00,38.71,4.00,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,118.70,22.76,1.00,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,boleto,1.00,141.46,4.00,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,159.90,19.22,1.00,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,credit_card,3.00,179.12,5.00,auto


## 11. Feature Engineering
Build the derived columns that EDA and ML will use.
All computed from columns already in `master` — no additional joins needed.

In [117]:
# delivery delay: positive = arrived late, negative = arrived early
master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] -
    master['order_estimated_delivery_date']
).dt.days

# total delivery duration from purchase to doorstep
master['delivery_days'] = (
    master['order_delivered_customer_date'] -
    master['order_purchase_timestamp']
).dt.days

# freight as a proportion of order value (how much of the spend is shipping?)
master['freight_ratio'] = (
    master['freight_value'] / master['order_value'].replace(0, np.nan)
).round(4)

# time features for seasonality EDA and ML
master['purchase_year']    = master['order_purchase_timestamp'].dt.year
master['purchase_month']   = master['order_purchase_timestamp'].dt.month
master['purchase_weekday'] = master['order_purchase_timestamp'].dt.dayofweek  # 0=Mon
master['purchase_hour']    = master['order_purchase_timestamp'].dt.hour

print('New columns added:')
new_cols = ['delivery_delay_days','delivery_days','freight_ratio',
            'purchase_year','purchase_month','purchase_weekday','purchase_hour']
print(master[new_cols].describe().round(2).to_string())

New columns added:
       delivery_delay_days  delivery_days  freight_ratio  purchase_year  purchase_month  purchase_weekday  purchase_hour
count            97,005.00      97,005.00      99,214.00      99,992.00       99,992.00         99,992.00      99,992.00
mean                -11.88          12.10           0.31       2,017.54            6.03              2.76          14.77
std                  10.18           9.55           0.31           0.51            3.23              1.97           5.33
min                -147.00           0.00           0.00       2,016.00            1.00              0.00           0.00
25%                 -17.00           6.00           0.13       2,017.00            3.00              1.00          11.00
50%                 -12.00          10.00           0.22       2,018.00            6.00              3.00          15.00
75%                  -7.00          15.00           0.38       2,018.00            8.00              4.00          19.00
max          

## 12. Validation Checks
Simple assertions to catch any data quality issues before saving.
If any check fails, fix it before moving on.

In [118]:
checks = {
    'order_id is unique':          master['order_id'].is_unique,
    'order_id has no nulls':       master['order_id'].notna().all(),
    'order_value > 0 (where set)': (master['order_value'].dropna() > 0).all(),
    'review_score in 1-5':         master['review_score'].dropna().between(1, 5).all(),
    'delivery_days >= 0 (where set)': (master['delivery_days'].dropna() >= 0).all(),
    'freight_ratio >= 0 (where set)': (master['freight_ratio'].dropna() >= 0).all(),
}

all_passed = True
for check, result in checks.items():
    status = 'PASS' if result else 'FAIL'
    print(f'  [{status}]  {check}')
    if not result:
        all_passed = False

print()
print('All checks passed!' if all_passed else 'Some checks FAILED — fix before saving.')

  [FAIL]  order_id is unique
  [PASS]  order_id has no nulls
  [PASS]  order_value > 0 (where set)
  [PASS]  review_score in 1-5
  [PASS]  delivery_days >= 0 (where set)
  [PASS]  freight_ratio >= 0 (where set)

Some checks FAILED — fix before saving.


## 13. Quick Summary Stats

In [119]:
print('=== MASTER DATAFRAME SUMMARY ===')
print(f'Total orders:          {len(master):,}')
print(f'Total columns:         {master.shape[1]}')
print(f'Date range:            {master["order_purchase_timestamp"].min().date()} '
      f'to {master["order_purchase_timestamp"].max().date()}')
print(f'Unique customers:      {master["customer_id"].nunique():,}')
print(f'Unique sellers:        {master["seller_id"].nunique():,}')
print(f'Unique categories:     {master["category_english"].nunique():,}')
print(f'Total GMV:             R$ {master["order_value"].sum():,.0f}')
print(f'Avg order value:       R$ {master["order_value"].mean():,.2f}')
print(f'Avg review score:      {master["review_score"].mean():.2f}')
print(f'Avg delivery days:     {master["delivery_days"].mean():.1f}')
print(f'Orders delivered late: {(master["delivery_delay_days"] > 0).sum():,} '
      f'({(master["delivery_delay_days"] > 0).mean()*100:.1f}%)')

=== MASTER DATAFRAME SUMMARY ===
Total orders:          99,992
Total columns:         27
Date range:            2016-09-04 to 2018-10-17
Unique customers:      99,441
Unique sellers:        3,088
Unique categories:     72
Total GMV:             R$ 13,651,923
Avg order value:       R$ 137.60
Avg review score:      4.09
Avg delivery days:     12.1
Orders delivered late: 6,563 (6.6%)


## 14. Save to CSV
Save `master` as a CSV. Every subsequent notebook loads this single file —
no need to re-run the joins and cleaning again.

In [120]:
import os
os.makedirs('../processed', exist_ok=True)

SAVE_PATH = '../processed/master_orders.csv'
master.to_csv(SAVE_PATH, index=False)

size_mb = os.path.getsize(SAVE_PATH) / 1024 / 1024
print(f'Saved to: {SAVE_PATH}')
print(f'File size: {size_mb:.1f} MB')
print(f'Shape: {master.shape}')

Saved to: ../processed/master_orders.csv
File size: 31.5 MB
Shape: (99992, 27)
